In [54]:
import json
import os
from collections.abc import Callable
from concurrent.futures import ThreadPoolExecutor
from typing import Any, Literal

from dotenv import load_dotenv
from google.genai import Client
from pydantic import BaseModel, Field, ValidationError

load_dotenv()

True

In [56]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

google_client = Client(api_key=GOOGLE_API_KEY)
MODEL = "gemini-3.5-flash-lite"

### Task 1

In [3]:
def get_weather(city: str, units: str = "celsius") -> dict:
    """
    Get the current weather for a given city.

    Args:
        city (str): The name of the city to get the weather for.
        units (str): The units to display the temperature in.

    Returns:
        dict: A dictionary containing the weather information.
    """

    return {
        "city": city,
        "temperature": "15°C" if units == "celsius" else "59°F",
        "description": "Partly cloudy",
    }


# hand made tool definition for OpenAI and Anthropic models

openai_tool_definition = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get the current weather for a given city. eg. New York, London, Paris",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city to get the weather for.",
                },
                "units": {
                    "type": "string",
                    "description": "The units to display the temperature in. Can be 'celsius' or 'fahrenheit'.",
                },
            },
            "required": ["city"],
        },
    }
]


anthropic_tool_definition = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a given city. eg. New York, London, Paris",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city to get the weather for.",
                },
                "units": {
                    "type": "string",
                    "description": "The units to display the temperature in. Can be 'celsius' or 'fahrenheit'.",
                },
            },
            "required": ["city"],
        },
    }
]

In [22]:
# tool definition from pydantic model for OpenAI and Anthropic models


class GetWeatherArgs(BaseModel):
    city: str = Field(
        description="The name of the city to get the weather for.eg. New York, London, Paris"
    )
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="The units to display the temperature in. Can be 'celsius' or 'fahrenheit'.",
    )


schema = GetWeatherArgs.model_json_schema()
print(schema)

{'properties': {'city': {'description': 'The name of the city to get the weather for.eg. New York, London, Paris', 'title': 'City', 'type': 'string'}, 'units': {'default': 'celsius', 'description': "The units to display the temperature in. Can be 'celsius' or 'fahrenheit'.", 'enum': ['celsius', 'fahrenheit'], 'title': 'Units', 'type': 'string'}}, 'required': ['city'], 'title': 'GetWeatherArgs', 'type': 'object'}


In [33]:
class ToolSpec(BaseModel):
    name: str
    description: str
    args_model: type[BaseModel]
    handler: Callable[..., Any]


WEATHER_TOOL_SPEC = ToolSpec(
    name="get_weather",
    description="Get the current weather for a given city. eg. New York, London, Paris",
    args_model=GetWeatherArgs,
    handler=get_weather,
)

In [35]:
def get_openai_tool_definition(tool_spec: ToolSpec) -> dict:
    schema = tool_spec.args_model.model_json_schema()
    return {
        "type": "function",
        "name": tool_spec.name,
        "description": tool_spec.description,
        "parameters": schema,
        "strict": True,
    }


def get_anthropic_tool_definition(tool_spec: ToolSpec) -> dict:
    schema = tool_spec.args_model.model_json_schema()
    return {
        "name": tool_spec.name,
        "description": tool_spec.description,
        "parameters": schema,
        "strict": True,
    }

In [36]:
print(json.dumps(get_openai_tool_definition(WEATHER_TOOL_SPEC), indent=4))

{
    "type": "function",
    "name": "get_weather",
    "description": "Get the current weather for a given city. eg. New York, London, Paris",
    "parameters": {
        "properties": {
            "city": {
                "description": "The name of the city to get the weather for.eg. New York, London, Paris",
                "title": "City",
                "type": "string"
            },
            "units": {
                "default": "celsius",
                "description": "The units to display the temperature in. Can be 'celsius' or 'fahrenheit'.",
                "enum": [
                    "celsius",
                    "fahrenheit"
                ],
                "title": "Units",
                "type": "string"
            }
        },
        "required": [
            "city"
        ],
        "title": "GetWeatherArgs",
        "type": "object"
    },
    "strict": true
}


In [37]:
print(json.dumps(get_anthropic_tool_definition(WEATHER_TOOL_SPEC), indent=4))

{
    "name": "get_weather",
    "description": "Get the current weather for a given city. eg. New York, London, Paris",
    "parameters": {
        "properties": {
            "city": {
                "description": "The name of the city to get the weather for.eg. New York, London, Paris",
                "title": "City",
                "type": "string"
            },
            "units": {
                "default": "celsius",
                "description": "The units to display the temperature in. Can be 'celsius' or 'fahrenheit'.",
                "enum": [
                    "celsius",
                    "fahrenheit"
                ],
                "title": "Units",
                "type": "string"
            }
        },
        "required": [
            "city"
        ],
        "title": "GetWeatherArgs",
        "type": "object"
    },
    "strict": true
}


### Task 2

In [ ]:
def validate_function_call_args(args_model: type[BaseModel], args: dict) -> BaseModel:
    """
    Validate the arguments for a given tool specification.

    Args:
        args_model (BaseModel): The argument model to validate against.
        args (dict): The arguments to validate.

    Returns:
        BaseModel: An instance of the tool's argument model with validated data.
    """

    try:
        raw_args = json.loads(args)
    except json.JSONDecodeError as e:
        return {"success": False, "error": f"Malformed JSON: {e}"}

    try:
        validated_args = args_model.model_validate(raw_args)
    except Exception as e:  # noqa: BLE001
        return {"success": False, "error": f"Validation error: {e}"}

    return {"success": True, "validated_args": validated_args}

In [45]:
# check with true argument
print(validate_function_call_args(GetWeatherArgs, '{"city": "New York", "units": "celsius"}'))

{'success': True, 'validated_args': GetWeatherArgs(city='New York', units='celsius')}


In [46]:
# malformed json argument
print(
    validate_function_call_args(GetWeatherArgs, '{"city": "New York", "units": "celsius"')
)  # Missing closing brace

{'success': False, 'error': "Malformed JSON: Expecting ',' delimiter: line 1 column 40 (char 39)"}


In [47]:
# validation error argument
print(
    validate_function_call_args(GetWeatherArgs, '{"city": "New York", "units": "kelvin"}')
)  # Invalid units

{'success': False, 'error': "Validation error: 1 validation error for GetWeatherArgs\nunits\n  Input should be 'celsius' or 'fahrenheit' [type=literal_error, input_value='kelvin', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/literal_error"}


### Task 3

In [50]:
tool_calls = [
    {"id": "call_1", "name": "get_weather", "args": {"city": "London"}},
    {"id": "call_2", "name": "get_weather", "args": {"city": "Paris"}},
    {"id": "call_3", "name": "get_weather", "args": {"city": "Tokyo"}},
]


def execute_tool(call):
    result = get_weather(**call["args"])

    return {
        "tool_call_id": call["id"],  # preserve ID
        "result": result,
    }


with ThreadPoolExecutor(max_workers=3) as executor:
    results = list(executor.map(execute_tool, tool_calls))


# Match results back by ID
results_by_id = {result["tool_call_id"]: result["result"] for result in results}

print(results_by_id)

{'call_1': {'city': 'London', 'temperature': '15°C', 'description': 'Partly cloudy'}, 'call_2': {'city': 'Paris', 'temperature': '15°C', 'description': 'Partly cloudy'}, 'call_3': {'city': 'Tokyo', 'temperature': '15°C', 'description': 'Partly cloudy'}}


### Task 4

In [57]:
class Person(BaseModel):
    name: str
    age: int
    city: str


test_set = [
    "John is 25 and lives in London.",
    "Alice, age 30, lives in Paris.",
    "Bob is 42 years old and lives in Tokyo.",
]


# -----------------------------
# Prompt-based JSON extraction
# -----------------------------
def prompt_json_extraction(text):
    interaction = google_client.interactions.create(
        model=MODEL,
        input=f"""
        Extract name, age and city.
        Return ONLY valid JSON.

        Text: {text}
        """,
    )

    return interaction.output_text


def extract_person(name, age, city):
    return {"name": name, "age": age, "city": city}


TOOL_DEFINITIONS = [
    {
        "type": "function",
        "name": "extract_person",
        "description": "Extract a person's name, age and city.",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "Person's name"},
                "age": {"type": "integer", "description": "Person's age"},
                "city": {"type": "string", "description": "Person's city"},
            },
            "required": ["name", "age", "city"],
        },
    }
]


# -----------------------------
# Forced tool extraction
# -----------------------------
def tool_extraction(text):
    interaction = google_client.interactions.create(
        model=MODEL,
        input=text,
        tools=TOOL_DEFINITIONS,
        generation_config={
            "tool_choice": {"allowed_tools": {"mode": "any", "tools": ["extract_person"]}}
        },
    )

    # Get function call
    tool_call = next(step for step in interaction.steps if step.type == "function_call")

    return tool_call.arguments


# -----------------------------
# Validation
# -----------------------------
def valid_json(output):
    try:
        Person.model_validate(json.loads(output))
        return True
    except (json.JSONDecodeError, ValidationError, TypeError):
        return False


def valid_tool(output):
    try:
        Person.model_validate(output)
        return True
    except ValidationError:
        return False


# -----------------------------
# Comparison
# -----------------------------
prompt_bad = 0
tool_bad = 0

for text in test_set:
    prompt_output = prompt_json_extraction(text)
    tool_output = tool_extraction(text)

    if not valid_json(prompt_output):
        prompt_bad += 1

    if not valid_tool(tool_output):
        tool_bad += 1


total = len(test_set)

print(f"Prompt JSON malformed rate : {prompt_bad / total:.2%}")
print(f"Forced Tool malformed rate : {tool_bad / total:.2%}")

Prompt JSON malformed rate : 0.00%
Forced Tool malformed rate : 0.00%


### Task 5

observed in task 2

### Task 6

yes verfied in task 2

### Task 7

In [62]:
WRONG_WEATHER_TOOL = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get the weather for a specific city.",
        "parameters": {
            "type": "object",
            "properties": {
                # bug: handler expects "city"
                # but schema tells model to use "loc"
                "loc": {
                    "type": "string",
                    "description": "The name of the city to get the weather for.",
                },
                "units": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["loc"],
        },
    }
]

interaction = google_client.interactions.create(
    model=MODEL,
    input="What is the weather in London?",
    tools=WRONG_WEATHER_TOOL,
    generation_config={"tool_choice": {"allowed_tools": {"mode": "any", "tools": ["get_weather"]}}},
)

for step in interaction.steps:
    print("TYPE:", step.type)

    if step.type == "function_call":
        print("NAME:", step.name)
        print("ARGUMENTS:", step.arguments)

TYPE: thought
TYPE: function_call
NAME: get_weather
ARGUMENTS: {'loc': 'London'}


In [63]:
tool_call = next(step for step in interaction.steps if step.type == "function_call")

result = get_weather(**tool_call.arguments)

TypeError: get_weather() got an unexpected keyword argument 'loc'

In [64]:
CORRECT_WEATHER_TOOL = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get the weather for a specific city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city to get the weather for.",
                },
                "units": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["city"],
        },
    }
]


interaction = google_client.interactions.create(
    model=MODEL,
    input="What is the weather in London?",
    tools=CORRECT_WEATHER_TOOL,
    generation_config={"tool_choice": {"allowed_tools": {"mode": "any", "tools": ["get_weather"]}}},
)

tool_call = next(step for step in interaction.steps if step.type == "function_call")

print(tool_call.arguments)

{'city': 'London'}


In [65]:
result = get_weather(**tool_call.arguments)

print(result)

{'city': 'London', 'temperature': '15°C', 'description': 'Partly cloudy'}
